Coreference resolution (2002-2025, for Libération, Le Monde, Le Figaro)

In [2]:
import os
import json
import pprint
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
from adjustText import adjust_text
import statistics
from matplotlib_venn import venn3
import textwrap

In [9]:
def corefs_by_year(path, papers):
    years = os.listdir(path)
    res = {}
    for year in years:
        y = year.split('.')[0]
        res[y] = {}
        fpath = os.path.join(path, year)
        with open(fpath) as d:
            data = json.load(d)
        for paper in papers:
            if paper in data:
                old = res[y]
                to_add = {k.lower(): v for k,v in data[paper].items()}
                new = {k: old.get(k, 0) + to_add.get(k, 0) for k in set(old) | set(to_add)}
                res[y] = new
        final = {k: v for k,v in res[y].items() if v>1}
        res[y] = final
    #print(json.dumps(res, sort_keys=True, indent=4))
    pprint.pp(res, indent=1, sort_dicts=True)


def corefs_by_paper(path, papers, crop=False):
    years = os.listdir(path)
    res = {paper: {} for paper in papers}
    for year in years:
        fpath = os.path.join(path, year)
        with open(fpath) as d:
            data = json.load(d)
        for paper in papers:
            if paper in data:
                old = res[paper]
                to_add = {k.lower().replace('etat', 'état'): v for k,v in data[paper].items()}
                new = {k: old.get(k, 0) + to_add.get(k, 0) for k in set(old) | set(to_add)}
                res[paper] = new
    for paper in papers:
        final = {k: v for k,v in res[paper].items() if v>1}
        if crop:
            sorted_final = {key: value for i, (key, value) in enumerate(sorted(final.items(), key=lambda item: item[1], reverse=True)) if i<10}
        else:
            sorted_final = {key: value for (key, value) in sorted(final.items(), key=lambda item: item[1], reverse=True)}
        res[paper] = sorted_final
    #print(json.dumps(res, sort_keys=True, indent=4))
    #pprint.pp(res)
    return res


def format_text(elements, width=15):
    #return "\n".join(textwrap.fill(e, width) for e in elements)
    return "\n".join('"'+e+'"' for e in elements)

def venn(res):
    dic = {}
    for k,v in res.items():
        dic[k] = set(word.replace('etat', 'état') for word in v.keys())
    values = list(dic.values())
    plt.figure(figsize=(10, 10))
    diag = venn3(values, dic.keys())
    zones = {
        '100': values[0] - values[1] - values[2],
        '010': values[1] - values[0] - values[2],
        '001': values[2] - values[0] - values[1],
        '110': (values[0] & values[1]) - values[2],
        '101': (values[0] & values[2]) - values[1],
        '011': (values[1] & values[2]) - values[0],
        '111': values[0] & values[1] & values[2]
    }
    for zone_id, elements in zones.items():
        label = diag.get_label_by_id(zone_id)
        if label:
            label.set_text(format_text(elements))
            label.set_fontsize(10)
            label.set_bbox(dict(facecolor='white', alpha=0.7, edgecolor='none'))
    colors = ['#1f77b4', '#d62728', '#2ca02c']
    for patch, color in zip([diag.get_patch_by_id('100'),
                         diag.get_patch_by_id('010'),
                         diag.get_patch_by_id('001')], colors):
        if patch:
            patch.set_color(color)
            patch.set_alpha(0.4)
    plt.show()

def unic_words(res):
    unics = {k: {} for k in res}
    for key,value in res.items():
        others = []
        for key2,value2 in res.items():
            if key2 != key:
                new = [k for k in set(others) | set(value2)]
                others = new
        unic = {k: value.get(k,0) for k in set(value) if k not in set(others)}
        sorted_unic = {key: value for i, (key, value) in enumerate(sorted(unic.items(), key=lambda item: item[1], reverse=True))}
        unics[key] = sorted_unic
    #print(json.dumps(unics, sort_keys=True, indent=4))
    pprint.pp(unics, sort_dicts=True)

SARKORPUS_LIGHT

In [ ]:
path = os.path.join('coref_resolution', 'all_corefs_v2')
papers = ['Le Figaro', 'Libération', 'Le Monde']

res2 = corefs_by_paper(path, papers, crop=True)
pprint.pp(res2)
venn(res2)

In [ ]:
res_unic = corefs_by_paper(path, papers)
unic_words(res_unic)

In [10]:
def prop_ex(path, papers):
    res = {}
    years = os.listdir(path)
    for year in years:
        an = int(year.split('.')[0])
        res[an] = {}
        for paper in papers:
            res[an][paper] = {'ex_pres': 0, 'total_pres': 0, 'sarko': 0, 'nicolas': 0}
        fpath = os.path.join(path, year)
        with open(fpath) as d:
            data = json.load(d)
        for k, v in data.items():
            for occ, n in v.items():
                occ = occ.lower()
                if ('président' in occ) or ('chef' in occ):
                    res[an][k]['total_pres'] += n
                    if ('ex' in occ) or ('ancien' in occ):
                        res[an][k]['ex_pres'] += n
                if occ == 'sarko':
                    res[an][k]['sarko'] += n
                if occ == 'nicolas':
                    res[an][k]['nicolas'] += n
    return res

In [11]:
def print_prop_ex(res, papers):
    colors = {
        'Le Figaro': '#1f77b4',
        'Le Monde': '#2ca02c',
        'Libération': '#d62728',
    }
    offsets = {
        'Le Figaro': -0.2,
        'Le Monde': 0,
        'Libération': 0.2,
    }
    years = sorted(res.keys())
    plt.figure(figsize=(12,6))
    ax = plt.gca()
    texts = []
    for paper in papers:
        xvals = [year + offsets.get(paper, 0) for year in years]
        vals = [(res[year][paper]['ex_pres']/res[year][paper]['total_pres'] if res[year][paper]['total_pres'] != 0 else np.nan) for year in years]
        n_pres = [(res[year][paper]['total_pres'] if res[year][paper]['total_pres'] != 0 else np.nan) for year in years]
        sizes = [np.sqrt(n)*40 if not np.isnan(n) else 0 for n in n_pres]
        plt.scatter(xvals, vals, s=sizes, color=colors.get(paper, None), label=paper, marker='o', alpha=0.8)
        for x, y, n in zip(xvals, vals, n_pres):
            if not np.isnan(y) and n>2:
                texts.append(
                    ax.text(x, y+ 0.015*np.nanmax(vals), n, fontsize=9, color=colors[paper])
                )
        mean = statistics.mean([v for v,y in zip(vals, years) if y>2012 and not np.isnan(v)])
        ax.plot([2012.5, 2025.5],
            [mean, mean],
            color=colors.get(paper, None), 
            linestyle='--', 
            linewidth=1.5, 
            label=paper+' ' + f'mean = {mean:.2f}')
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    plt.xlabel("Year")
    plt.ylabel("Proportions")
    #plt.title("Proportion of 'ex', 'ancien' when calling Sarkozy president (2002-2025)")
    plt.grid(True, axis='x', linestyle='--', alpha=0.4)
    plt.legend(
        title="Newspaper",
        frameon=False,
        ncol=1
    )
    plt.tight_layout()
    plt.show()

In [12]:
def print_nom(res, papers, key):
    colors = {
        'Le Figaro': '#1f77b4',
        'Le Monde': '#2ca02c',
        'Libération': '#d62728',
    }
    offsets = {
        'Le Figaro': -0.2,
        'Le Monde': 0,
        'Libération': 0.2,
    }
    years = sorted(res.keys())
    plt.figure(figsize=(12,6))
    ax = plt.gca()
    texts = []
    for paper in papers:
        xvals = [year + offsets.get(paper, 0) for year in years]
        vals = [res[year][paper][key] for year in years]
        sizes = [n*20 for n in vals]
        plt.scatter(xvals, vals, s=sizes, color=colors.get(paper, None), label=paper, marker='o', alpha=0.8)
        for x, y, n in zip(xvals, vals, vals):
            if n>1:
                texts.append(
                    ax.text(x, y+ 0.015*np.nanmax(vals), n, fontsize=9, color=colors[paper])
                )
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    plt.xlabel("Year")
    plt.ylabel("Number of mention")
    #plt.title("Number of time a paper called Nicolas Sarkozy '" + key.capitalize() +"' (2002-2025)")
    plt.grid(True, axis='x', linestyle='--', alpha=0.4)
    plt.legend(
        title="Newspaper",
        frameon=False,
        ncol=1,
        loc='upper left'
    )
    plt.tight_layout()
    plt.show()

In [6]:
papers = ['Libération', 'Le Monde', 'Le Figaro']
path2 = os.path.join('coref_resolution', 'all_corefs_v2')
results2 = prop_ex(path2, papers)

In [ ]:
print_prop_ex(results2, papers)

In [ ]:
for k,v in res2.items():
    print(k)
    nsarko = v['sarko']
    nsarkozy = v['sarkozy']
    psarko = nsarko/nsarkozy
    print(f'Prop = {psarko:.2f}', f'n sarko = {nsarko}', f'n sarkozy = {nsarkozy}')
    print()

print_nom(results2, papers, 'sarko')

SARKORPUS_BIG_TITLES

In [ ]:
path = os.path.join('coref_resolution_title', 'all_corefs_v2')
papers = ['Le Figaro', 'Libération', 'Le Monde']

res = corefs_by_paper(path, papers, crop=True)
pprint.pp(res)
venn(res)

In [ ]:
res_unic = corefs_by_paper(path, papers)
unic_words(res_unic)

In [16]:
papers = ['Libération', 'Le Monde', 'Le Figaro']
path = os.path.join('coref_resolution_title', 'all_corefs_v2')
results = prop_ex(path, papers)

In [ ]:
print_prop_ex(results, papers)

In [ ]:
res = corefs_by_paper(path, papers, crop=False)
for k,v in res.items():
    print(k)
    nsarko = v.get('sarko', 0)
    nsarkozy = v['sarkozy']
    psarko = nsarko/nsarkozy
    print(f'Prop = {psarko:.2f}', f'n sarko = {nsarko}', f'n sarkozy = {nsarkozy}')
    print()

print_nom(results, papers, 'sarko')